# TextSeal: Post-hoc Watermarking Demo

This notebook demonstrates how to use [TextSeal](https://github.com/facebookresearch/textseal) to watermark text using LLM rephrasing.

**What this notebook does:**
1. Install textseal from PyPI
2. Watermark your text with a detectable watermark
3. Verify the watermark was successfully embedded

[Documentation](https://github.com/facebookresearch/textseal/blob/main/docs/README_posthoc_api.md) | [Paper](https://arxiv.org/abs/2512.16904)

## Setup

Install textseal and provide your input text:

In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = "https://github.com/facebookresearch/textseal"
repo_dir = Path("textseal")

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url], check=True)

os.chdir(repo_dir)
print(Path.cwd())

In [ ]:
%pip install -U pip
%pip install -e .
%pip install huggingface_hub
%pip install transformers==5.3.0

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

**Input options:** Edit text directly below, upload a file (`UPLOAD_FILE = True`), or provide a file path.

In [2]:
# =============================================================================
# OPTION A: Direct text input (edit this)
# =============================================================================
TEXT_INPUT = """
The sun rose over the quiet village, casting long shadows across the cobblestone streets.
Birds chirped in the distance as shopkeepers prepared their stalls for the day.
Children hurried to school, their laughter echoing through the morning air.
By noon, the marketplace was bustling with activity, filled with the aroma of fresh bread and spices.
""".strip()

# =============================================================================
# OPTION B: Upload a file (set to True to enable)
# =============================================================================
UPLOAD_FILE = False

# =============================================================================
# OPTION C: File path (local or mounted drive)
# =============================================================================
FILE_PATH = None  # e.g., "/content/drive/MyDrive/document.txt"

# =============================================================================
# Load text based on selected option
# =============================================================================
if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    text = uploaded[filename].decode('utf-8')
    print(f"Loaded from uploaded file: {filename}")
elif FILE_PATH:
    with open(FILE_PATH, 'r') as f:
        text = f.read()
    print(f"Loaded from file: {FILE_PATH}")
else:
    text = TEXT_INPUT
    print("Using direct text input")

print(f"\nText length: {len(text)} characters")
print(f"\n--- Preview (first 500 chars) ---\n{text[:500]}")

Using direct text input

Text length: 347 characters

--- Preview (first 500 chars) ---
The sun rose over the quiet village, casting long shadows across the cobblestone streets.
Birds chirped in the distance as shopkeepers prepared their stalls for the day.
Children hurried to school, their laughter echoing through the morning air.
By noon, the marketplace was bustling with activity, filled with the aroma of fresh bread and spices.


## Post-hoc Watermarking

Configure and create the watermarker:

In [3]:
from textseal import PostHocWatermarker, WatermarkConfig, ModelConfig, ProcessingConfig

# Configuration
MODEL_NAME = "Qwen/Qwen3.5-2B"  # Options: "gumbelmax", "greenlist", "synthid", etc.
WATERMARK_TYPE = "textseal"  # Options: "gumbelmax", "greenlist", "synthid", etc.
MIXING_ALPHA = 0.5  # Probability of using Key A per token (0.5 = equal)
TEMPERATURE = 0.9  # Higher = stronger watermark but lower text quality
SECRET_KEY = 42 

# Create watermarker
watermarker = PostHocWatermarker(
    watermark_config=WatermarkConfig(
        watermark_type=WATERMARK_TYPE,
        secret_key=SECRET_KEY,
        mixing_alpha=MIXING_ALPHA,
    ),
    model_config=ModelConfig(
        model_name=MODEL_NAME,
    ),
    processing_config=ProcessingConfig(temperature=TEMPERATURE),
    verbose=True,
)

print(f"\nWatermarker ready!")
print(f"   - Watermark type: {WATERMARK_TYPE}")
print(f"   - Model: {MODEL_NAME}")
print(f"   - Temperature: {TEMPERATURE}")

/home/pfz/miniconda3/envs/text_seal_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer and model: Qwen/Qwen3.5-2B
✓ Tokenizer loaded successfully


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 1702.32it/s, Materializing param=model.norm.weight]                              


✓ Model Qwen/Qwen3.5-2B loaded successfully


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1894.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Watermarker ready!
   - Watermark type: textseal
   - Model: Qwen/Qwen3.5-2B
   - Temperature: 0.9


Apply the watermark:

In [4]:
# Watermark the text
print("Watermarking text (this may take a moment)...\n")
result = watermarker.process_text(text)

# Display results
print("=" * 60)
print("WATERMARKING COMPLETE")
print("=" * 60)

print(f"\nDetection Results:")
print(f"   - P-value: {result['wm_eval']['p_value']:.2e}")
print(f"   - Detected: {'Yes' if result['wm_eval']['det'] else 'No'}")
print(f"   - Score: {result['wm_eval']['score']:.4f}")

print(f"\nStatistics:")
print(f"   - Original tokens: {result['stats']['orig_toks']}")
print(f"   - Watermarked tokens: {result['stats']['wm_toks']}")
print(f"   - Token ratio: {result['stats']['tok_ratio']:.2f}")

print(f"\nTiming:")
print(f"   - Total time: {result['times']['t_total']:.2f}s")
print(f"   - Tokens/sec: {result['times']['tps']:.1f}")

Watermarking text (this may take a moment)...

WATERMARKING COMPLETE

Detection Results:
   - P-value: 1.32e-05
   - Detected: Yes
   - Score: 1.3781

Statistics:
   - Original tokens: 71
   - Watermarked tokens: 78
   - Token ratio: 1.10

Timing:
   - Total time: 9.18s
   - Tokens/sec: 8.6


View original vs watermarked text:

In [5]:
print("=" * 60)
print("ORIGINAL TEXT")
print("=" * 60)
print(text)

print("\n" + "=" * 60)
print("WATERMARKED TEXT")
print("=" * 60)
print(result["wm_text"])

ORIGINAL TEXT
The sun rose over the quiet village, casting long shadows across the cobblestone streets.
Birds chirped in the distance as shopkeepers prepared their stalls for the day.
Children hurried to school, their laughter echoing through the morning air.
By noon, the marketplace was bustling with activity, filled with the aroma of fresh bread and spices.

WATERMARKED TEXT
As the sun crested the horizon, painting the tranquil village with extended shadows over the cobblestones. Echoing melodies drifted from the birds as vendors arranged their stalls for the day. Children darted toward school, their giggles rippling through the early morning atmosphere. By midday, the market buzzed with vigor, brimming with the scent of fresh baked goods and exotic spices.


### Verify

Verify watermark detection (works on any text with the same secret key):

In [6]:
# Verify the watermark
verification = watermarker.evaluate_watermark(result["wm_text"])

print("Verification Results:")
print(f"   - P-value: {verification['p_value']:.2e}")
print(f"   - Detected: {'Yes' if verification['det'] else 'No'}")
print(f"   - Score: {verification['score']:.4f}")

# Also test original (unwatermarked) text
print("\nOriginal text (should NOT be detected):")
orig_verification = watermarker.evaluate_watermark(text)
print(f"   - P-value: {orig_verification['p_value']:.2e}")
print(f"   - Detected: {'Yes' if orig_verification['det'] else 'No'}")

Verification Results:
   - P-value: 1.32e-05
   - Detected: Yes
   - Score: 1.3781

Original text (should NOT be detected):
   - P-value: 5.75e-01
   - Detected: No


**Note on p-value, Detection, and False Positive Rate**

- **P-value**: The p-value measures the probability that we obtain a higher score than the observed watermark detection score by random chance in unwatermarked text. A lower p-value therefore indicates stronger evidence that the watermark is present.
- **Detection**: Detection is typically reported as "Yes" if the p-value falls below a chosen threshold (here, 0.001).
- **False Positive Rate**: The detection threshold (such as p < 0.001) directly controls the false positive rate (the probability of incorrectly detecting a watermark in unwatermarked text). For example, a threshold of 0.001 means that, on average, 0.1% of unwatermarked texts may be falsely flagged as watermarked.

---

## Generation-time Watermarking

In **generation mode** (`generation_mode=True`), the watermark is applied during *initial generation* — the model treats your text as a prompt and produces a fresh watermarked response from scratch, rather than rephrasing existing text. This is the natural setting when you are generating new content and want the watermark embedded at creation time.


In [7]:
# Switch the existing watermarker to generation mode (reuses the already-loaded model)
watermarker.processing_config.generation_mode = True
watermarker.processing_config.max_gen_len = 100

PROMPT = "Explain the importance of watermarking AI-generated text in two paragraphs."

print("Generation mode enabled.")
print(f"Prompt: {PROMPT}")


Generation mode enabled.
Prompt: Explain the importance of watermarking AI-generated text in two paragraphs.


In [ ]:
print("Generating watermarked response...\n")
gen_result = watermarker.process_text(PROMPT)

print("=" * 60)
print("GENERATION COMPLETE")
print("=" * 60)

print(f"\nPrompt:\n{PROMPT}")
print(f"\nGenerated (watermarked) response:\n{gen_result['wm_text']}")

print(f"\nDetection Results:")
print(f"   - P-value: {gen_result['wm_eval']['p_value']:.2e}")
print(f"   - Detected: {'Yes' if gen_result['wm_eval']['det'] else 'No'}")
print(f"   - Score: {gen_result['wm_eval']['score']:.4f}")
print(f"\nTiming: {gen_result['times']['t_total']:.2f}s  ({gen_result['times']['tps']:.1f} tok/s)")


Generating watermarked response...

GENERATION COMPLETE

Prompt:
Explain the importance of watermarking AI-generated text in two paragraphs.

Generated (watermarked) response:
Watermarking AI-generated text serves as a crucial authentication mechanism that distinguishes human creativity from synthetic content, thereby protecting intellectual property and maintaining market integrity. In an era where deepfakes, forged credentials, and plagiarized work are prevalent, a visible or invisible marker acts as a quick verification tool for authors, platforms, and consumers. Without such a feature, it becomes increasingly difficult to trace the source of an original idea or text, leading to the erosion of legal rights and the potential for cybercrime

Detection Results:
   - P-value: 0.000000
   - Detected: Yes
   - Score: 1.6086

Timing: 12.00s  (8.4 tok/s)


### Localized Detection

Standard detection tests the *entire* text for a watermark signal. **Localized detection** goes further: it searches for the sub-region with the *strongest* concentrated signal using a geometric cover scan over power-of-two window sizes. It also returns per-token labels (via a sliding-window smoother) that indicate which tokens fall in high-scoring regions.

| Field | Meaning |
|---|---|
| `global_pvalue` | p-value over the entire text |
| `localized_pvalue` | p-value of the best sub-region (Bonferroni-corrected) |
| `final_pvalue` | min(global, localized) × 2 — combined test |
| `region_start/end` | indices of the best sub-region in the scored token sequence |
| `token_labels` | per-scored-token label: 1 = in high-scoring region |


In [9]:
from textseal.watermarking.detector import localized_detect
from IPython.display import HTML, display

LOREM_IPSUM = """
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi commodo ut est et facilisis. Morbi sodales cursus commodo. Integer mollis libero lorem, sit amet pharetra neque convallis sed. Morbi elementum est eu quam iaculis tristique. Donec id turpis vitae ex pellentesque aliquam ac sit amet felis. Cras interdum, mi vitae iaculis euismod, metus lacus semper diam, eu viverra lacus justo id sem. Vivamus pellentesque ante at laoreet euismod.
Nunc dictum pharetra placerat. Curabitur et velit non est porttitor aliquam posuere a orci. Proin laoreet purus turpis, vulputate molestie tortor semper quis. Phasellus blandit tortor a dolor euismod, ut lacinia felis cursus. Duis ut viverra enim. Cras euismod a elit sed convallis. Donec ornare porta porta. Etiam sit amet pharetra sem. In fermentum fermentum libero. Etiam pretium, elit quis efficitur tincidunt, enim lacus vulputate mi, vel bibendum augue nisi vitae elit. Vivamus ornare magna nunc, ac sagittis velit dignissim eget. Duis sed pulvinar urna, at aliquet erat. Curabitur pellentesque erat porta erat ullamcorper, et volutpat magna fermentum. Duis quis rhoncus tortor.
Phasellus interdum mi risus, in iaculis massa ultrices sed. Duis fermentum tellus a tincidunt finibus. Cras elementum nunc eu dolor convallis pulvinar. Etiam et tincidunt ante. Vivamus rhoncus, ligula non accumsan tristique, leo nibh aliquam arcu, in tincidunt metus est vitae massa. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Quisque imperdiet mattis ligula eget pulvinar.
Cras laoreet pellentesque libero et fermentum. Sed elementum magna sed elit cursus rutrum. Duis aliquet, velit vel aliquet facilisis, sem quam gravida eros, vitae semper sem lorem a nisl. Proin gravida vitae nulla a porttitor. Sed hendrerit metus vestibulum luctus dignissim. Vivamus ornare volutpat augue ut pellentesque. Fusce tempus dui non arcu maximus varius. Vestibulum vehicula mollis neque et tristique. Quisque elementum metus vel erat blandit, quis facilisis urna sagittis. Nullam convallis porta lorem, ac lobortis purus. Maecenas venenatis lectus hendrerit, dignissim elit in, tristique odio.
Proin convallis ultrices laoreet. Quisque fringilla felis pretium justo tincidunt hendrerit. Duis vel tempor turpis. Integer varius velit quis pellentesque eleifend. Aliquam ullamcorper maximus libero, at ultrices lacus ultrices in. Sed efficitur augue in quam placerat sagittis. Ut pharetra ligula in leo placerat accumsan eget interdum odio.
"""

# Embed the watermarked text between two blocks of unwatermarked lorem ipsum.
mixed_text = LOREM_IPSUM[:300] + "\n\n" + gen_result["wm_text"] + "\n\n" + LOREM_IPSUM[300:]

loc = localized_detect(
    mixed_text,
    tokenizer=watermarker.tokenizer,
    wm_config=watermarker.watermark_config,
    model=watermarker.model,
)

print("Localized Detection (watermarked text embedded inside lorem ipsum):")
print(f"  Global p-value:     {loc.global_pvalue:.2e}")
print(f"  Localized p-value:  {loc.localized_pvalue:.2e}")
print(f"  Final p-value:      {loc.final_pvalue:.2e}")
print(f"  Detected:           {'Yes ✓' if loc.detected else 'No ✗'}")
print(f"  Scored tokens:      {loc.n_tokens}")
print(f"  Strongest region:   scored token indices {loc.region_start}–{loc.region_end}")

watermark_frac = sum(loc.token_labels) / max(1, len(loc.token_labels))
print(f"\n  Watermarked token fraction (boundary smoother): {watermark_frac:.1%}")

# ASCII heatmap over scored (deduplicated) positions.
if loc.token_labels:
    bar_width = 60
    bar = "".join(
        "█" if loc.token_labels[int(ii / bar_width * len(loc.token_labels))] == 1 else "░"
        for ii in range(bar_width)
    )
    print(f"  Token heatmap (█ = watermarked region, ░ = lorem ipsum):")
    print(f"  [{bar}]")

# --- Correct position→label mapping ---
# localized_detect deduplicates (context, token) pairs, so token_labels[i] maps to
# the i-th *unique* (ctx, tok) occurrence — not simply token[ngram+1+i].
# Replicate the same dedup loop to find the exact token position each label belongs to.
ngram = watermarker.watermark_config.ngram
token_ids = watermarker.tokenizer.encode(mixed_text, add_special_tokens=False)
seen = set()
scored_positions = []
for pos in range(ngram + 1, len(token_ids)):
    ctx = tuple(token_ids[pos - ngram:pos])
    tok = token_ids[pos]
    dedup_key = ctx + (tok,)
    if dedup_key in seen:
        continue
    seen.add(dedup_key)
    scored_positions.append(pos)

# Map each scored position to its label (unscored positions default to 0).
pos_to_label = {pos: loc.token_labels[ii] for ii, pos in enumerate(scored_positions)}
aligned_labels = [pos_to_label.get(pos, 0) for pos in range(len(token_ids))]

# HTML per-token highlight over the full mixed text.
parts = []
for tok_id, lbl in zip(token_ids, aligned_labels):
    word = watermarker.tokenizer.decode([tok_id]).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    color = "#90EE90" if lbl == 1 else "transparent"
    parts.append(f'<span style="background:{color}">{word}</span>')

display(HTML(
    '<div style="font-family:monospace;line-height:2;padding:12px;border:1px solid #ccc;border-radius:6px">'
    '<b>Per-token watermark signal</b> (green = detected watermarked region, plain = lorem ipsum):<br><br>'
    + "".join(parts)
    + "</div>"
))


Localized Detection (watermarked text embedded inside lorem ipsum):
  Global p-value:     6.64e-03
  Localized p-value:  5.35e-10
  Final p-value:      1.07e-09
  Detected:           Yes ✓
  Scored tokens:      629
  Strongest region:   scored token indices 64–192

  Watermarked token fraction (boundary smoother): 22.7%
  Token heatmap (█ = watermarked region, ░ = lorem ipsum):
  [░░░░░░░██████████░█░░░░░░░░░░░░░░░░░░░░░░░░░░██░░░░░░░░░░░░░]


Save results (optional):

In [10]:
import json

# Save watermarked text
with open("watermarked_text.txt", "w") as f:
    f.write(result["wm_text"])

# Save full results as JSON
with open("watermark_results.json", "w") as f:
    json.dump({
        "original_text": result["orig_text"],
        "watermarked_text": result["wm_text"],
        "detection": result["wm_eval"],
        "stats": result["stats"],
        "config": {
            "watermark_type": WATERMARK_TYPE,
            "model": MODEL_NAME,
            "temperature": TEMPERATURE,
            "secret_key": SECRET_KEY,
        }
    }, f, indent=2)

print("Saved:")
print("   - watermarked_text.txt")
print("   - watermark_results.json")

# Download files (Colab)
try:
    from google.colab import files
    files.download("watermarked_text.txt")
    files.download("watermark_results.json")
except:
    pass

Saved:
   - watermarked_text.txt
   - watermark_results.json
